In [1]:
import random
from tqdm import tqdm
from collections import defaultdict
def get_available_nodes(graph, source, hop=1):
    if hop == 0:
        return set(graph.keys()) - {source}
    current = {source}
    visited = {source}
    available_nodes = set()
    for _ in range(hop):
        next_level = set()
        for node in current:
            neighbors = graph[node]
            available_nodes.update(neighbors)
            for neighbor in neighbors:
                if neighbor not in visited:
                    visited.add(neighbor)
                    next_level.add(neighbor)
        current = next_level
        if not current:
            break
    available_nodes.discard(source)
    return available_nodes

def simulate(graph, victim, originator, available_nodes, p=1):
    if originator not in available_nodes:
        raise ValueError("Originator not in victim neighborhood")
    victim_neighbors = graph[victim]
    remaining = set(victim_neighbors)  # not yet infected
    current = {originator}
    visited = {originator}
    time = 0
    spread_time = 0
    if originator in remaining:
        remaining.remove(originator)

    while current and remaining:
        next_level = set()
        for node in current:
            neighbors = available_nodes & graph[node]
            for neighbor in neighbors:
                if neighbor not in visited and random.random() <= p:
                    visited.add(neighbor)
                    next_level.add(neighbor)
                    if neighbor in remaining:
                        remaining.remove(neighbor)
                        spread_time = time + 1  # last infection time updates
        current = next_level
        time += 1
    victim_degree = len(victim_neighbors)
    infected = victim_neighbors - remaining
    spread_factor = len(infected) / victim_degree if victim_degree > 0 else 0

    return spread_time, spread_factor

def run_simulation_by_k(graph, hop=1, p=1):
    data_by_k = defaultdict(lambda: defaultdict(list))
    for victim, victim_node in graph.items():
        available_nodes = get_available_nodes(graph, victim, hop=hop)
        for originator in victim_node:
            data = simulate(graph, victim, originator, available_nodes, p)
            spread_time, spread_factor = data[0], data[1]
            k = len(victim_node)
            data_by_k[k]["spread_time"].append(spread_time)
            data_by_k[k]["spread_factor"].append(spread_factor)
    return data_by_k

In [2]:
import pickle
import os
def load_graphs(path):
    with open(path, "rb") as f:
        data = pickle.load(f)
    return data

def save_results(results, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "wb") as f:
        pickle.dump(results, f)

def to_dict(obj):
    if isinstance(obj, defaultdict):
        return {k: to_dict(v) for k, v in obj.items()}
    return obj

In [3]:
size = 10
for p in range(20):
    if not os.path.exists(f"fig_data/fig07/fig07_p{p:02d}.pkl"):
        result = defaultdict(lambda: defaultdict(list))
        small_world_path = f"data/small_world/sw_network_p{p:02d}_N{size:03d}K.pkl"
        sw_graphs = load_graphs(small_world_path)
        print(f"Running simulation for p={p:02d}:")
        for graph in tqdm(sw_graphs):
            graph_result = run_simulation_by_k(graph)
            for k, metrics in graph_result.items():
                for metric, values in metrics.items():
                    result[k][metric].extend(values)
        result = to_dict(result)
        del sw_graphs
        save_results(result, f"fig_data/fig07/fig07_p{p:02d}.pkl")
        del result

Running simulation for p=00:


100%|██████████| 500/500 [00:47<00:00, 10.55it/s]


Running simulation for p=01:


100%|██████████| 500/500 [00:49<00:00, 10.13it/s]


Running simulation for p=02:


...

100%|██████████| 500/500 [00:35<00:00, 13.90it/s]


Running simulation for p=18:


100%|██████████| 500/500 [00:32<00:00, 15.48it/s]


Running simulation for p=19:


100%|██████████| 500/500 [00:31<00:00, 15.71it/s]


In [4]:
size = 1
for p in range(3):
    if not os.path.exists(f"fig_data/fig07/fig07_random_p{p:02d}.pkl"):
        result = defaultdict(lambda: defaultdict(list))
        random_path = f"data/random/random_network_p{p:02d}_N{size:03d}K.pkl"
        random_graphs = load_graphs(random_path)
        print(f"Running simulation for p={p:02d}:")
        for graph in tqdm(random_graphs):
            graph_result = run_simulation_by_k(graph)
            for k, metrics in graph_result.items():
                for metric, values in metrics.items():
                    result[k][metric].extend(values)
        result = to_dict(result)
        del random_graphs
        save_results(result, f"fig_data/fig07/fig07_random_p{p:02d}.pkl")
        del result

Running simulation for p=00:


100%|██████████| 500/500 [00:27<00:00, 17.99it/s]


Running simulation for p=01:


100%|██████████| 500/500 [06:17<00:00,  1.33it/s]


Running simulation for p=02:


100%|██████████| 500/500 [1:39:57<00:00, 12.00s/it]


In [7]:
import networkx as nx

def compute_graph_metrics(graph):
    G = nx.Graph(graph)
    C = nx.average_clustering(G)
    L = nx.average_shortest_path_length(G)
    return C, L

In [ ]:
def generate_small_world_network(num_nodes, k, p):
    graph = {i: set() for i in range(num_nodes)}

    # Initial ring lattice
    for i in range(num_nodes):
        for j in range(1, k // 2 + 1):
            neighbor = (i + j) % num_nodes
            graph[i].add(neighbor)
            graph[neighbor].add(i)
    nodes = list(range(num_nodes))

    # Rewire
    for i in range(num_nodes):
        for j in range(1, k // 2 + 1):
            if random.random() > p:
                continue
            old_neighbor = (i + j) % num_nodes
            # Remove old edge
            graph[i].discard(old_neighbor)
            graph[old_neighbor].discard(i)
            # Avoid self-loops and duplicate edges
            forbidden = set(graph[i])
            forbidden.add(i)

            while True:
                new_neighbor = random.choice(nodes)
                if new_neighbor not in forbidden:
                    break

            graph[i].add(new_neighbor)
            graph[new_neighbor].add(i)
    return graph

graph0 = generate_small_world_network(num_nodes=10000, k=4, p=-1)
C0, L0 = compute_graph_metrics(graph0)
print(f"C0={C0:.4f}, L0={L0:.4f}")

size = 10
for p in range(20):
    if not os.path.exists(f"fig_data/fig07/fig07_p{p:02d}_CL.pkl"):
        result = defaultdict(list)
        small_world_path = f"data/small_world/sw_network_p{p:02d}_N{size:03d}K.pkl"
        sw_graphs = load_graphs(small_world_path)
        print(f"Computing metrics for p={p:02d}:")
        for graph in tqdm(sw_graphs):
            C, L = compute_graph_metrics(graph)
            result["C"].append(C)
            result["L"].append(L)
        result = to_dict(result)
        del sw_graphs
        save_results(result, f"fig_data/fig07/fig07_p{p:02d}_CL.pkl")
        del result

C0=0.5000, L0=1250.3750
Computing metrics for p=00:


  2%|▏         | 8/500 [06:07<6:24:20, 46.87s/it]